# Assignment 3

In [2]:
#Practice Push 

In [3]:
import pandas as pd
import ast

df = pd.read_csv("track_data_final.csv")

# Parse album release date and extract year
df["album_release_date"] = pd.to_datetime(df["album_release_date"], errors="coerce")
df["year"] = df["album_release_date"].dt.year

# Convert duration to minutes and round to 2 decimals
df["duration_min"] = (df["track_duration_ms"] / 60000.0).round(2)

# Parse artist_genres string -> list
def parse_genres(x):
    if pd.isna(x) or x == "" or x == "[]":
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df["genre_list"] = df["artist_genres"].apply(parse_genres)

# Main genre = first genre, or "Unknown"
def main_genre(genres):
    if genres:
        return genres[0]
    return "Unknown"

df["main_genre"] = df["genre_list"].apply(main_genre)

# Select dashboard columns
dash_cols = [
    "track_name", "artist_name", "album_name", "album_type",
    "track_popularity", "artist_popularity",
    "duration_min", "explicit", "year", "main_genre"
]

dash_df = df[dash_cols].dropna(subset=["track_popularity", "artist_popularity", "year"])

# Ensure integer types for popularity and year
dash_df["track_popularity"] = dash_df["track_popularity"].astype(int)
dash_df["artist_popularity"] = dash_df["artist_popularity"].astype(int)
dash_df["year"] = dash_df["year"].astype(int)

dash_df.head(20)


,track_name,artist_name,album_name,album_type,track_popularity,artist_popularity,duration_min,explicit,year,main_genre
0,3,Britney Spears,The Singles Collection,compilation,61,80,3.55,False,2009,pop
1,Clouds,BUNT.,Clouds,single,67,69,2.65,False,2023,stutter house
2,Forever & Always (Taylor’s Version),Taylor Swift,Fearless (Taylor's Version),album,63,100,3.76,False,2021,Unknown
3,I Didn't Change My Number,Billie Eilish,Happier Than Ever,album,72,90,2.64,True,2021,Unknown
4,Man Down,Rihanna,Loud,album,57,90,4.45,False,2010,Unknown
5,Space Song,Beach House,Depression Cherry,album,77,72,5.34,False,2015,dream pop
6,idontwannabeyouanymore,Billie Eilish,dont smile at me,single,78,90,3.39,False,2017,Unknown
7,Allein Allein - BENNETT Remix,Alok,Allein Allein (feat. FREY) [BENNETT Remix],single,52,76,2.43,False,2024,brazilian bass
8,Even My Dad Does Sometimes,Ed Sheeran,x (Deluxe Edition),album,50,88,3.81,False,2014,soft pop
9,Eyes Blue Like The Atlantic (feat. Subvrbs),Sista Prod,Eyes Blue Like The Atlantic (feat. Subvrbs),single,63,48,2.58,False,2020,Unknown


In [4]:
import panel as pn
import hvplot.pandas

pn.extension('tabulator')  # or pn.extension() if this errors

In [6]:
import panel as pn
import hvplot.pandas

pn.extension('tabulator')

# Widgets
genre_widget = pn.widgets.MultiSelect(
    name="Genre",
    options=sorted(dash_df["main_genre"].unique()),
    value=["pop"],
    size=8
)

year_slider = pn.widgets.IntRangeSlider(
    name="Year range",
    start=int(dash_df["year"].min()),
    end=int(dash_df["year"].max()),
    value=(int(dash_df["year"].min()), int(dash_df["year"].max()))
)

# Plain Python function to filter the DataFrame
def filter_df(genres, year_range):
    dff = dash_df.copy()
    if genres:
        dff = dff[dff["main_genre"].isin(genres)]
    if year_range and len(year_range) == 2:
        dff = dff[(dff["year"] >= year_range[0]) & (dff["year"] <= year_range[1])]
    return dff

# Bound, reactive DataFrame
filtered = pn.bind(filter_df, genres=genre_widget, year_range=year_slider)

# 1) Scatter: track vs artist popularity
scatter = pn.bind(
    lambda df: df.hvplot.scatter(
        x="artist_popularity",
        y="track_popularity",
        by="main_genre",
        hover_cols=["track_name", "artist_name", "album_name", "album_type", "year"],
        title="Track vs Artist Popularity",
        height=350, width=450,
    ),
    df=filtered
)

# 2) Bar: average popularity by genre
def make_bar(df):
    genre_mean = (
        df.groupby("main_genre")["track_popularity"]
        .mean()
        .sort_values(ascending=False)
        .head(12)
    )
    if genre_mean.empty:
        return genre_mean.hvplot.bar().opts(title="No data for selected filters")
    return genre_mean.hvplot.bar(
        rot=45,
        ylabel="Avg track popularity",
        xlabel="Genre",
        title="Average Track Popularity by Genre",
        height=350, width=450,
    )

bar = pn.bind(make_bar, df=filtered)

# 3) Duration vs album type (box plot)
duration_box = pn.bind(
    lambda df: df.hvplot.box(
        y="duration_min",
        by="album_type",
        ylabel="Duration (min)",
        xlabel="Album type",
        title="Track Duration by Album Type",
        height=350, width=450,
    ),
    df=filtered
)

# 4) Popularity trend over time
def make_trend(df):
    year_mean = df.groupby("year")["track_popularity"].mean()
    if year_mean.empty:
        return year_mean.hvplot.line().opts(title="No data for selected filters")
    return year_mean.hvplot.line(
        ylabel="Avg track popularity",
        xlabel="Year",
        title="Popularity Over Time",
        height=350, width=450,
    )

trend = pn.bind(make_trend, df=filtered)

# Layout with Panel
controls = pn.Column("## Filters", genre_widget, year_slider)

plots = pn.Column(
    pn.Row(scatter, bar),
    pn.Row(duration_box, trend),
)

dashboard = pn.Row(controls, plots)

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'dfb8f59e-a4bd-4517-9a60-a3f8877086fd': {'version…